# Portfolio - Optimization & Allocation

Notebook scaffold for portfolio optimization and allocation analysis.

In [ ]:
# Code Block 1: Notebook description

# Focused scaffold for optimization inputs only.

# Outputs from this notebook: invested tickers and directional signage.


In [ ]:
#paramters
period = "max"
interval = "1d"

In [ ]:
# Code Block 2: Load Libraries

# Load Libraries

import numpy as np

import pandas as pd



import statsmodels

import statsmodels.api as sm

from statsmodels.tsa.stattools import coint

from IPython.display import display


from schwab.auth import easy_client



import os

import matplotlib.pyplot as plt

import plotly.express as px

from plotly.subplots import make_subplots

import plotly.graph_objects as go

import sys

from pathlib import Path
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()




from Quantapp.data import yf as qa_yf
from Quantapp.data import get_schwab_portfolio_snapshot
from Quantapp.data.adapters import SCHWAB_OPTION_SYMBOL_PATTERN


from Quantapp.data import MacroDataClient

from Quantapp.secrets import load_project_env, require_secret



load_project_env()




qe = MacroDataClient()

In [ ]:
# Code Block 3: Direction/sign helpers



OPTION_PATTERN = SCHWAB_OPTION_SYMBOL_PATTERN



def normalize_yf_ticker(ticker):

    if not isinstance(ticker, str):

        return ticker

    return ticker.strip().replace('/', '-')



def build_yf_ticker_map(tickers):

    return {ticker: normalize_yf_ticker(ticker) for ticker in tickers}



def infer_directional_signage(positions):

    option_rows = []



    for position in positions:

        instrument = position.get("instrument", {})

        if instrument.get("assetType") != "OPTION":

            continue



        raw_symbol = "".join((instrument.get("symbol", "") or "").split())

        match = OPTION_PATTERN.match(raw_symbol)

        if not match:

            continue



        option_rows.append(

            {

                "underlying": match.group("underlying"),

                "option_type": match.group("option_type"),

                "long_quantity": position.get("longQuantity", 0.0),

                "short_quantity": position.get("shortQuantity", 0.0),

                "average_price": position.get("averagePrice", 0.0),

            }

        )



    positions_df = pd.DataFrame(option_rows)

    if positions_df.empty:

        return positions_df, pd.Series(dtype=object), pd.Series(dtype=float)



    direction_factor = positions_df["option_type"].map({"C": 1.0, "P": -1.0}).fillna(0.0)

    positions_df["net_quantity"] = positions_df["long_quantity"] - positions_df["short_quantity"]

    positions_df["directional_value"] = positions_df["net_quantity"] * positions_df["average_price"] * direction_factor * 100



    net_direction = positions_df.groupby("underlying")["directional_value"].sum()

    sentiment_map = net_direction.apply(lambda x: "bullish" if x > 0 else ("bearish" if x < 0 else "neutral"))

    directional_signage = sentiment_map.map({"bullish": 1, "bearish": -1, "neutral": 0}).astype(int)



    return positions_df, sentiment_map, directional_signage


In [ ]:
# Code Block 4: Define Schwab parameters

CLIENT_ID = require_secret("SCHWAB_CLIENT_ID")
APP_SECRET = require_secret("SCHWAB_APP_SECRET")
CALLBACK_URL = os.getenv("SCHWAB_CALLBACK_URL", "https://127.0.0.1:8182")

_token_path = os.getenv("SCHWAB_TOKEN_PATH")
TOKEN_PATH = Path(_token_path).expanduser() if _token_path else PROJECT_ROOT / "schwab_token.json"
if not TOKEN_PATH.is_absolute():
    TOKEN_PATH = PROJECT_ROOT / TOKEN_PATH
TOKEN_PATH.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
# Code Block 5: Login to Schwab client
from schwab.auth import easy_client

client = easy_client(
    api_key=CLIENT_ID,
    app_secret=APP_SECRET,
    callback_url=CALLBACK_URL,
    token_path=str(TOKEN_PATH),
)

print(f"Schwab client ready. Token cache: {TOKEN_PATH}")


In [ ]:
# Code Block 6: Retrieve account and market data
# Schwab account retrieval and position normalization live in Quantapp.data.
portfolio_snapshot = get_schwab_portfolio_snapshot(client)

account_information = portfolio_snapshot.account_information
acct_map = portfolio_snapshot.account_numbers
acct_hash = portfolio_snapshot.account_hash
acct = portfolio_snapshot.account
positions = portfolio_snapshot.raw_positions
positions_df = portfolio_snapshot.option_positions
option_sentiment = portfolio_snapshot.option_sentiment
net_direction = portfolio_snapshot.net_direction
organized_positions = portfolio_snapshot.organized_positions
invested_symbols = portfolio_snapshot.invested_symbols
net_invested_amounts = portfolio_snapshot.net_invested_amounts
total_margin = portfolio_snapshot.total_margin
option_pattern = SCHWAB_OPTION_SYMBOL_PATTERN

# Retrieve core market data for benchmark and portfolio.
benchmark_data = qa_yf.Ticker('SPY').history(period=period, interval=interval)
invested_symbol_map = build_yf_ticker_map(invested_symbols)

if invested_symbol_map:
    portfolio_data = qa_yf.download(
        tickers=list(invested_symbol_map.values()),
        period=period,
        interval=interval,
        auto_adjust=True,
        threads=True,
        progress=False,
    )
    portfolio_closing_prices = portfolio_data['Close']
    if isinstance(portfolio_closing_prices, pd.Series):
        portfolio_closing_prices = portfolio_closing_prices.to_frame(name=invested_symbols[0])
    else:
        portfolio_closing_prices = portfolio_closing_prices.rename(
            columns={yf_ticker: ticker for ticker, yf_ticker in invested_symbol_map.items()}
        )
else:
    portfolio_closing_prices = pd.DataFrame(index=benchmark_data.index)

benchmark_close = benchmark_data['Close']
portfolio_closing_prices.index = portfolio_closing_prices.index.tz_localize(None)
benchmark_close.index = benchmark_close.index.tz_localize(None)

invested_tickers = invested_symbols
directional_signage = net_direction['sign'] if 'sign' in net_direction else pd.Series(dtype=int)
directional_signage_table = directional_signage.rename('sign').reset_index().rename(columns={'underlying': 'ticker'})

display(directional_signage_table.sort_values('ticker').reset_index(drop=True))
raw_prices = portfolio_closing_prices.copy()


In [ ]:
# Mean-Variance Optimization (Max Sharpe) from raw_prices
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# 1) Build return series
returns = raw_prices.pct_change().dropna(how="all")
returns = returns.dropna(axis=1, how="all") # remove dead columns

# Optional: align to invested_tickers if you want strict set
if "invested_tickers" in globals():
    cols = [c for c in invested_tickers if c in returns.columns]
    returns = returns[cols]

# 2) Estimate mu and Sigma (annualized)
trading_days = 252
mu = returns.mean() * trading_days
Sigma = returns.cov() * trading_days

n = len(mu)
if n == 0:
    raise ValueError("No valid return series found for optimization.")

# Risk-free rate assumption (adjust as needed)
rf = 0.04

# 3) Max Sharpe objective
def neg_sharpe(w, mu, Sigma, rf):
    port_ret = w @ mu.values
    port_vol = np.sqrt(w @ Sigma.values @ w)
    if port_vol <= 0:
        return 1e6
    return -((port_ret - rf) / port_vol)

# Constraints
cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
bounds = [(0.0, 1.0)] * n # long-only
w0 = np.repeat(1.0 / n, n)

res = minimize(
    neg_sharpe,
    w0,
    args=(mu, Sigma, rf),
    method="SLSQP",
    bounds=bounds,
    constraints=cons,
)

if not res.success:
    raise RuntimeError(f"Optimization failed: {res.message}")

weights = pd.Series(res.x, index=mu.index, name="weight").sort_values(ascending=False)

# Diagnostics
opt_ret = float(weights.values @ mu.values)
opt_vol = float(np.sqrt(weights.values @ Sigma.values @ weights.values))
opt_sharpe = (opt_ret - rf) / opt_vol

display(weights.to_frame())
print(f"Expected Return: {opt_ret:.2%}")
print(f"Expected Vol: {opt_vol:.2%}")
print(f"Expected Sharpe: {opt_sharpe:.3f}")

# 4) Plot optimized weights
plot_weights = weights[weights > 0].sort_values(ascending=True)

plt.figure(figsize=(10, max(4, 0.35 * len(plot_weights))))
plt.barh(plot_weights.index, plot_weights.values)
plt.title("Optimized Portfolio Weights (Max Sharpe)")
plt.xlabel("Weight")
plt.ylabel("Ticker")
plt.xlim(0, max(0.05, plot_weights.max() * 1.15))
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

# 5) Plot random portfolios and highlight optimizer solution
num_portfolios = 3000
rand_w = np.random.dirichlet(np.ones(n), size=num_portfolios)
rand_ret = rand_w @ mu.values
rand_vol = np.sqrt(np.einsum("ij,jk,ik->i", rand_w, Sigma.values, rand_w))
rand_sharpe = (rand_ret - rf) / rand_vol

plt.figure(figsize=(10, 6))
sc = plt.scatter(rand_vol, rand_ret, c=rand_sharpe, cmap="viridis", s=10, alpha=0.6)
plt.colorbar(sc, label="Sharpe")
plt.scatter([opt_vol], [opt_ret], color="red", s=120, marker="*", label="Optimal (Max Sharpe)")
plt.title("Mean-Variance Opportunity Set")
plt.xlabel("Annualized Volatility")
plt.ylabel("Annualized Return")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()